# Ablation comparison

Read-only until every validation passes. This notebook reads the six variant `study.json`, `summary.json`, and `candidate_status.csv` files, verifies that the shared study identity and expected ablation switches match, then writes `<STUDY_ROOT>/comparison.csv`.

It reports candidate detections only, not accuracy, precision, recall, or IoU.

## Kaggle setup

1. In **Settings -> Internet**, turn internet **on**.
2. Add the Kaggle secret `huggingface_token` with read access to the output dataset repository.
3. Run this notebook only after all six run notebooks have finished.
4. Set `ALLOW_PARTIAL = True` only when you deliberately need a partial diagnostic report.

In [ ]:
%pip install -q "huggingface_hub>=0.23"

In [ ]:
from __future__ import annotations

import csv
import io
import json
import os
import random
import time
from pathlib import Path

from huggingface_hub import HfApi, hf_hub_download

STUDY_ROOT = "ablations/study_01"
OUTPUT_REPO = "sasudo2/landslides"
OUTPUT_REVISION = "main"
ALLOW_PARTIAL = False
COMPARISON_FILENAME = "comparison.csv"

STUDY_SCHEMA_VERSION = "ablation-study-v1"
OUTPUT_SCHEMA_VERSION = "ablation-output-v1"
VARIANTS = {
    "baseline": {"modality": "fused", "slope": True, "slic": True, "bonferroni": True},
    "optical_only": {"modality": "optical", "slope": True, "slic": True, "bonferroni": True},
    "sar_only": {"modality": "sar", "slope": True, "slic": True, "bonferroni": True},
    "no_slope": {"modality": "fused", "slope": False, "slic": True, "bonferroni": True},
    "no_slic": {"modality": "fused", "slope": True, "slic": False, "bonferroni": True},
    "no_bonferroni": {"modality": "fused", "slope": True, "slic": True, "bonferroni": False},
}
SHARED_IDENTITY_KEYS = (
    "study_schema_version",
    "output_schema_version",
    "raw_repo",
    "raw_commit_sha",
    "source_csv_name",
    "source_csv_sha256",
    "cohort",
    "incident_ids",
    "required_raw_inputs",
    "before_file_by_incident",
    "settings",
    "package_versions",
    "pipeline_sha256",
    "template_sha256",
)
COMPARISON_FIELDS = [
    "variant",
    "cohort_size",
    "n_successful",
    "n_failed",
    "n_incidents_with_candidates",
    "n_candidates",
    "candidates_per_successful_incident",
    "raw_commit_sha",
    "pipeline_sha256",
    "template_sha256",
    "study_identity_check",
    "run_complete",
    "switches",
]
TRANSIENT_STATUS_CODES = {408, 409, 412, 425, 429, 500, 502, 503, 504}


def resolve_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError as exc:
        raise RuntimeError("Set HF_TOKEN or run on Kaggle with the huggingface_token secret attached") from exc
    try:
        return UserSecretsClient().get_secret("huggingface_token")
    except Exception as exc:
        raise RuntimeError("Attach the Kaggle secret 'huggingface_token' with read access") from exc


HF_TOKEN = resolve_hf_token()
API = HfApi(token=HF_TOKEN)


def canonical_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def _status_code(exc):
    response = getattr(exc, "response", None)
    return getattr(response, "status_code", None)


def is_transient_hub_error(exc):
    status = _status_code(exc)
    if status in TRANSIENT_STATUS_CODES:
        return True
    if isinstance(exc, (ConnectionError, TimeoutError)):
        return True
    message = str(exc).lower()
    markers = (
        "timeout", "timed out", "temporarily", "connection reset", "connection aborted",
        "rate limit", "too many requests", "server error", "bad gateway",
        "service unavailable", "gateway timeout",
    )
    return any(marker in message for marker in markers) or status in TRANSIENT_STATUS_CODES


def retry_hub(action, description, attempts=5, base_delay=2.0):
    for attempt in range(1, attempts + 1):
        try:
            return action()
        except Exception as exc:
            if attempt >= attempts or not is_transient_hub_error(exc):
                raise
            delay = min(base_delay ** attempt, 60.0) + random.uniform(0.0, 1.0)
            print(f"{description} failed on attempt {attempt}/{attempts}: {exc}; retrying in {delay:.1f}s")
            time.sleep(delay)
    raise RuntimeError(f"Retry loop exhausted for {description}")


def repo_files():
    def list_tree():
        return list(API.list_repo_tree(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            revision=OUTPUT_REVISION,
            path_in_repo=study_root,
            recursive=True,
        ))

    try:
        entries = retry_hub(list_tree, f"list files under {OUTPUT_REPO}@{OUTPUT_REVISION}/{study_root}")
    except Exception as exc:
        status = _status_code(exc)
        message = str(exc).lower()
        if status == 404 or "not found" in message or "entry not found" in message:
            return set()
        raise
    files = set()
    for entry in entries:
        path = getattr(entry, "path", None)
        if path and (getattr(entry, "type", None) == "file" or hasattr(entry, "blob_id")):
            files.add(path)
    return files


def download(path):
    return retry_hub(
        lambda: hf_hub_download(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            revision=OUTPUT_REVISION,
            filename=path,
            token=HF_TOKEN,
        ),
        f"download {path}",
    )


def read_json(path):
    return json.loads(Path(download(path)).read_text(encoding="utf-8"))


def read_csv(path):
    with Path(download(path)).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


def derive_counts(study, summary, status_rows, name):
    ids = study["incident_ids"]
    errors = []
    if summary is not None:
        if summary.get("variant") != name:
            errors.append(f"{name}: summary variant is {summary.get('variant')!r}")
        try:
            counts = {
                "n_incidents": int(summary["n_incidents"]),
                "n_ok": int(summary["n_ok"]),
                "n_failed": int(summary["n_failed"]),
                "n_candidates": int(summary["n_candidates"]),
                "n_incidents_with_candidates": int(summary["n_incidents_with_candidates"]),
            }
        except (KeyError, TypeError, ValueError) as exc:
            errors.append(f"{name}: invalid summary.json ({exc})")
            counts = None
    else:
        counts = None
    if status_rows is not None:
        status_ids = []
        failed = 0
        ok = 0
        candidates = 0
        with_candidates = 0
        for row in status_rows:
            try:
                incident_id = int(row["incident_id"])
            except (KeyError, TypeError, ValueError) as exc:
                errors.append(f"{name}: invalid candidate_status.csv row ({exc})")
                continue
            status_ids.append(incident_id)
            if row.get("status") == "ok":
                ok += 1
                try:
                    n = int(row.get("n_candidates", 0))
                except (TypeError, ValueError):
                    n = 0
                candidates += n
                with_candidates += int(n > 0)
            else:
                failed += 1
        if status_ids != ids:
            errors.append(f"{name}: candidate_status.csv does not contain the study cohort in order")
        status_counts = {
            "n_incidents": len(status_rows),
            "n_ok": ok,
            "n_failed": failed,
            "n_candidates": candidates,
            "n_incidents_with_candidates": with_candidates,
        }
        if counts is None:
            counts = status_counts
        elif counts != status_counts:
            errors.append(f"{name}: summary.json and candidate_status.csv disagree")
    if counts is None:
        errors.append(f"{name}: summary.json and candidate_status.csv are both missing")
        counts = {
            "n_incidents": 0,
            "n_ok": 0,
            "n_failed": len(ids),
            "n_candidates": 0,
            "n_incidents_with_candidates": 0,
        }
    complete = (
        counts["n_incidents"] == len(ids)
        and counts["n_failed"] == 0
        and counts["n_ok"] == len(ids)
    )
    return counts, complete, errors


study_root = STUDY_ROOT.strip()
if not study_root or study_root.startswith("/") or ".." in study_root.split("/"):
    raise ValueError("STUDY_ROOT must be a relative study folder such as ablations/study_01")
while study_root.endswith("/"):
    study_root = study_root[:-1]

files = repo_files()
missing_folders = [
    name for name in VARIANTS
    if not any(path.startswith(f"{study_root}/{name}/") for path in files)
]
if missing_folders:
    raise FileNotFoundError(
        "Missing variant folders in " + f"{OUTPUT_REPO}@{OUTPUT_REVISION}: {missing_folders}"
    )

required_files = ("study.json", "run.json", "candidate_status.csv", "candidate_metadata.csv", "diagnostics.json", "summary.json")
studies = {}
summaries = {}
statuses = {}
problems = []
for name in VARIANTS:
    prefix = f"{study_root}/{name}/"
    present = {path[len(prefix):] for path in files if path.startswith(prefix)}
    study_path = prefix + "study.json"
    if "study.json" not in present:
        problems.append(f"{name}: missing study.json")
        continue
    study = read_json(study_path)
    if study.get("variant") != name:
        problems.append(f"{name}: study.json variant is {study.get('variant')!r}")
    if study.get("switches") != VARIANTS[name]:
        problems.append(f"{name}: unexpected ablation switches {study.get('switches')!r}")
    if study.get("study_schema_version") != STUDY_SCHEMA_VERSION:
        problems.append(f"{name}: unexpected study schema version")
    if study.get("output_schema_version") != OUTPUT_SCHEMA_VERSION:
        problems.append(f"{name}: unexpected output schema version")
    cohort = study.get("cohort", {})
    ids = study.get("incident_ids", [])
    if cohort.get("declaration") != "df.iloc[900:1000]" or cohort.get("start") != 900 or cohort.get("stop") != 1000:
        problems.append(f"{name}: unexpected cohort declaration {cohort!r}")
    if len(ids) != 100:
        problems.append(f"{name}: cohort has {len(ids)} incident IDs, expected 100")
    if len(set(ids)) != len(ids):
        problems.append(f"{name}: cohort contains duplicate incident IDs")
    studies[name] = study

    if ALLOW_PARTIAL:
        if "summary.json" not in present and "candidate_status.csv" not in present:
            problems.append(f"{name}: partial mode needs summary.json or candidate_status.csv")
        summary = read_json(prefix + "summary.json") if "summary.json" in present else None
        status_rows = read_csv(prefix + "candidate_status.csv") if "candidate_status.csv" in present else None
    else:
        absent = [filename for filename in required_files if filename not in present]
        if absent:
            problems.append(f"{name}: incomplete folder, missing {absent}")
        summary = read_json(prefix + "summary.json") if "summary.json" in present else None
        status_rows = read_csv(prefix + "candidate_status.csv") if "candidate_status.csv" in present else None
    run_record = read_json(prefix + "run.json") if "run.json" in present else None
    if not ALLOW_PARTIAL:
        if run_record is None:
            problems.append(f"{name}: missing run.json")
        elif run_record.get("status") != "complete":
            problems.append(f"{name}: run.json status is {run_record.get('status')!r}")
    summaries[name] = summary
    statuses[name] = status_rows

if problems:
    raise RuntimeError("Cannot compare runs yet: " + "; ".join(problems))

if not studies:
    raise RuntimeError("No readable study manifests")

first_name = next(iter(VARIANTS))
reference_name = first_name if first_name in studies else next(iter(studies))
reference = studies[reference_name]
reference_ids = reference["incident_ids"]
conflicts = []
for name, study in studies.items():
    for key in SHARED_IDENTITY_KEYS:
        if key not in study:
            conflicts.append(f"{name}: missing shared field {key}")
        elif canonical_json(study[key]) != canonical_json(reference[key]):
            conflicts.append(f"{name}: shared field {key} differs")
    if study.get("incident_ids") != reference_ids:
        conflicts.append(f"{name}: ordered cohort IDs differ")
if conflicts:
    raise RuntimeError("Cross-run identity check failed: " + "; ".join(conflicts))

rows = []
completeness_errors = []
for name, study in studies.items():
    counts, complete, errors = derive_counts(study, summaries.get(name), statuses.get(name), name)
    completeness_errors.extend(errors)
    if not complete and not ALLOW_PARTIAL:
        completeness_errors.append(f"{name}: run is not complete ({counts['n_failed']} failed incident(s))")
    rows.append({
        "variant": name,
        "cohort_size": len(study["incident_ids"]),
        "n_successful": counts["n_ok"],
        "n_failed": counts["n_failed"],
        "n_incidents_with_candidates": counts["n_incidents_with_candidates"],
        "n_candidates": counts["n_candidates"],
        "candidates_per_successful_incident": (counts["n_candidates"] / counts["n_ok"]) if counts["n_ok"] else 0.0,
        "raw_commit_sha": study["raw_commit_sha"],
        "pipeline_sha256": study["pipeline_sha256"],
        "template_sha256": study["template_sha256"],
        "study_identity_check": "pass",
        "run_complete": "yes" if complete else "no",
        "switches": json.dumps(study["switches"], sort_keys=True),
    })

if completeness_errors:
    raise RuntimeError(
        "Cannot write comparison.csv: " + "; ".join(completeness_errors)
        + ". Set ALLOW_PARTIAL = True to write a partial diagnostic report."
    )

buffer = io.StringIO(newline="")
writer = csv.DictWriter(buffer, fieldnames=COMPARISON_FIELDS)
writer.writeheader()
writer.writerows(rows)
comparison_bytes = buffer.getvalue().encode("utf-8")
comparison_path = f"{study_root}/{COMPARISON_FILENAME}"
retry_hub(
    lambda: API.upload_file(
        repo_id=OUTPUT_REPO,
        repo_type="dataset",
        revision=OUTPUT_REVISION,
        path_or_fileobj=io.BytesIO(comparison_bytes),
        path_in_repo=comparison_path,
        commit_message=f"Compare six ablations in {study_root}",
    ),
    f"upload {comparison_path}",
)

for row in rows:
    print(json.dumps(row, sort_keys=True))
print(f"Wrote {comparison_path}")